## Author: Chan Jin Wei
---

In [1]:
from pyspark.sql import SparkSession
import pandas as pd
from malaya.supervised.huggingface import load
from malaya.torch_model.huggingface import Classification
from classes.dataframe_saver import DataFrameSaver
import subprocess

/home/student/de-assgt/de-assgt/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/student/de-assgt/de-assgt/lib/python3.10/site-packages/malaya/tokenizer.py:214: FutureWarning: Possible nested set at position 3397
  self.tok = re.compile(r'({})'.format('|'.join(pipeline)))
/home/student/de-assgt/de-assgt/lib/python3.10/site-packages/malaya/tokenizer.py:214: FutureWarning: Possible nested set at position 3927
  self.tok = re.compile(r'({})'.format('|'.join(pipeline)))


In [2]:
subprocess.run(['hdfs', 'dfs', '-rm', 'dictionary4/*'])

Deleted dictionary4/_SUCCESS
Deleted dictionary4/dictionary4.csv


CompletedProcess(args=['hdfs', 'dfs', '-rm', 'dictionary4/*'], returncode=0)

In [3]:
spark = SparkSession.builder.appName("SentimentAnalysis").getOrCreate()

input_file = 'dictionary3/dictionary3.csv'
output_dir = '/home/student/de-assgt/content'
file_name = 'dictionary4.csv'
hdfs_path = 'dictionary4'

24/12/22 13:17:31 WARN Utils: Your hostname, LAPTOP-PFPL3CLD. resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
24/12/22 13:17:31 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
24/12/22 13:17:32 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
24/12/22 13:17:33 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


In [4]:
pyspark_df = spark.read.csv(input_file, header=True, inferSchema=True)

# Convert PySpark DataFrame to Pandas for processing
pandas_df = pyspark_df.toPandas()

# Define the label and HuggingFace model information
label = ['negative', 'neutral', 'positive']

available_huggingface = {
    'mesolitica/sentiment-analysis-nanot5-tiny-malaysian-cased': {
        'Size (MB)': 93,
        'macro precision': 0.67768,
        'macro recall': 0.68266,
        'macro f1-score': 0.67997,
    },
    'mesolitica/sentiment-analysis-nanot5-small-malaysian-cased': {
        'Size (MB)': 167,
        'macro precision': 0.67602,
        'macro recall': 0.67120,
        'macro f1-score': 0.67339,
    }
}

# Load the HuggingFace sentiment model
model_name = 'mesolitica/sentiment-analysis-nanot5-small-malaysian-cased'
sentiment_model = load(
    model=model_name,
    class_model=Classification,
    available_huggingface=available_huggingface,
    force_check=True,
    path=None,  # Use None or specify a valid path if required
)

# Perform sentiment analysis
def predict_sentiment(text):
    if pd.notnull(text):  # Check if text is not NaN
        return sentiment_model.predict([text])[0]
    return None

pandas_df['sentiment'] = pandas_df['words'].apply(predict_sentiment)

result_df = spark.createDataFrame(pandas_df)

DataFrameSaver.save_to_csv(result_df, output_dir, file_name)

result_df.show()


+-------------+--------------------+----------+-----------+----------+--------------------+--------------+-----------+---------------+---------+
|        words|          definition|   pos_tag|freq_of_use|kata_dasar|       kata_terbitan|num_variations|word_length|length_category|sentiment|
+-------------+--------------------+----------+-----------+----------+--------------------+--------------+-----------+---------------+---------+
|tengah-tengah|                   -|         -|          -|         -|                   -|             0|         13|           long|  neutral|
| menyampaikan|[me.nyam.pai.kan]...|         -|          -|         -|                   -|             0|         12|           long|  neutral|
| perkembangan|[per.kem.ba.ngan]...|kata kerja|   Moderate|   kembang|berkembang, menge...|             5|         12|           long|  neutral|
|  perkongsian|[per.kong.sian]  ...| kata nama|       High|    kongsi|berkongsi, perkon...|             2|         11|           l

In [5]:
result_df.coalesce(1).write.csv(hdfs_path, header=True, mode="overwrite")
print(f"Article content saved to HDFS at: {hdfs_path}")

Article content saved to HDFS at: dictionary4


In [6]:
subprocess.run(['hdfs', 'dfs', '-mv', 'dictionary4/part-00000*', 'dictionary4/dictionary4.csv'])

CompletedProcess(args=['hdfs', 'dfs', '-mv', 'dictionary4/part-00000*', 'dictionary4/dictionary4.csv'], returncode=0)

In [7]:
spark.stop()